In [ ]:
!pip install coremltools
import torch
import torch.nn as nn
import torch.optim as optim
import coremltools as ct
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Dataset
import kagglehub
import os

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Download 12 Class Dataset
print("Downloading 12-Class Dataset...")
path = kagglehub.dataset_download("mostafaabla/garbage-classification")

# Smart Path Finder
def find_correct_data_folder(start_path):
    print(f"Scanning for data in: {start_path}...")
    for root, dirs, files in os.walk(start_path):
        # Look for a folder that contains specific classes from the new dataset
        if "battery" in dirs and "biological" in dirs:
            print(f"Found dataset at: {root}")
            return root
    raise FileNotFoundError("Could not find the class folders!")

data_dir = find_correct_data_folder(path)

# Data Preprocessing with augmentation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, shear=10, scale=(0.8, 1.2)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class MapDataset(Dataset):
    def __init__(self, dataset, map_transform):
        self.dataset = dataset
        self.map_transform = map_transform
    def __getitem__(self, index):
        img, label = self.dataset[index]
        return self.map_transform(img), label
    def __len__(self):
        return len(self.dataset)

full_dataset = datasets.ImageFolder(root=data_dir, transform=None)
class_names = full_dataset.classes
print(f"Classes found ({len(class_names)}): {class_names}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_idx, val_idx = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(MapDataset(train_idx, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(MapDataset(val_idx, val_transform), batch_size=32)

# Model Setup
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

for param in model.features.parameters():
    param.requires_grad = False

# Update classifier for 12 classes
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, len(class_names))
)
model = model.to(device)

# Training Loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print("Starting Training (10 Epochs)...")
for epoch in range(10):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1} finished. Loss: {running_loss/len(train_loader):.4f}")

# Export
class DeploymentModel(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.softmax = nn.Softmax(dim=1)
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.mean) / self.std
        logits = self.model(x)
        return self.softmax(logits)

model_cpu = model.to("cpu")
deploy_model = DeploymentModel(model_cpu)
deploy_model.eval()

dummy_input = torch.rand(1, 3, 224, 224)
traced_model = torch.jit.trace(deploy_model, dummy_input)

print("Converting to CoreML...")
ml_model = ct.convert(
    traced_model,
    inputs=[ct.ImageType(name="image", shape=dummy_input.shape, scale=1/255.0)],
    classifier_config=ct.ClassifierConfig(class_names)
)

ml_model.save("GarbageClassifier.mlpackage")
print("Done! Download 'GarbageClassifier.mlpackage'")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 7.3 MB/s eta 0:00:00


Using device: cuda
Using Colab cache for faster access to the 'garbage-classification' dataset.
Scanning for data in: /kaggle/input/garbage-classification...
✅ Found dataset at: /kaggle/input/garbage-classification/garbage_classification
Classes found (12): ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 175MB/s]


Starting Training (10 Epochs)...
Epoch 1 finished. Loss: 0.8287
Epoch 2 finished. Loss: 0.4519
Epoch 3 finished. Loss: 0.3963
Epoch 4 finished. Loss: 0.3755
Epoch 5 finished. Loss: 0.3492
Epoch 6 finished. Loss: 0.3337
Epoch 7 finished. Loss: 0.3316
Epoch 8 finished. Loss: 0.3154
Epoch 9 finished. Loss: 0.3194
Epoch 10 finished. Loss: 0.3198


Converting to CoreML...


Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/usr/local/lib/python3.12/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '829', of the source model, has been renamed to 'var_829' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 168.78 passes/s]


DONE! Download 'GarbageClassifier_12Classes.mlpackage'
